# Authenticated UN Comtrade API Exploration

## Purpose

This notebook tests the authenticated UN Comtrade final-data endpoint using a subscription key.

The goals are to:
- verify authenticated access from Python;
- compare the authenticated response with the public preview response;
- test multiple periods and trade flows;
- inspect the returned monthly records;
- prepare the extraction logic for the production pipeline.

In [1]:
# Project Setup
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
subscription_key = os.getenv("COMTRADE_API_KEY")

# Check existence
if not subscription_key:
    raise ValueError(
        "COMTRADE_API_KEY was not found."
    )
    
print("API key loaded:", subscription_key is not None)

# Check type
print('Key type: ', type(subscription_key))

API key loaded: True
Key type:  <class 'str'>


## First authenticated request

This request verifies that the subscription key can be loaded from the
local environment and used by the reusable extraction function.

The request retrieves Germany's total merchandise exports to the World
for January 2023.

In [3]:
# Import fetching and saving functionalities
from src.extract import (
    fetch_trade_data,
    save_raw_payload,
)

# Test the API with an example
authenticated_payload = fetch_trade_data(
    type_code="C",
    frequency_code="M",
    classification_code="HS",
    period="202301",
    reporter_code="276",
    partner_code="0",
    commodity_code="TOTAL",
    flow_code="X",
    subscription_key=subscription_key,
    max_records=100_000,
)

print("Returned count:", authenticated_payload["count"])
print("API error:", authenticated_payload["error"])

first_record = authenticated_payload["data"][0]

print("Reporter:", first_record["reporterDesc"])
print("Period:", first_record["period"])
print("Flow:", first_record["flowDesc"])
print("Partner:", first_record["partnerDesc"])
print("Commodity:", first_record["cmdDesc"])
print("Primary value:", first_record["primaryValue"])

print('Payload type: ', type(authenticated_payload))

Returned count: 1
API error: 
Reporter: Germany
Period: 202301
Flow: Export
Partner: World
Commodity: All Commodities
Primary value: 139553698569.058
Payload type:  <class 'dict'>


## Multiple-period request

This request tests whether the authenticated final-data endpoint accepts
multiple comma-separated periods in one call.

The request retrieves Germany's total exports to the World for
January, February, and March 2023.

In [4]:
multi_period_payload = fetch_trade_data(
    type_code="C",
    frequency_code="M",
    classification_code="HS",
    period="202301,202302,202303",
    reporter_code="276",
    partner_code="0",
    commodity_code="TOTAL",
    flow_code="X",
    subscription_key=subscription_key,
    max_records=100_000,
)

print("Returned count:", multi_period_payload["count"])
print("API error:", multi_period_payload["error"])

for record in multi_period_payload["data"]:
    print(
        "Period:",
        record["period"],
        "| Flow:",
        record["flowDesc"],
        "| Value:",
        record["primaryValue"],
    )

Returned count: 3
API error: 
Period: 202301 | Flow: Export | Value: 139553698569.058
Period: 202302 | Flow: Export | Value: 142244471966.542
Period: 202303 | Flow: Export | Value: 163396316089.323


## Multiple-period and multiple-flow request

This request tests whether the authenticated endpoint accepts both
multiple periods and multiple trade flows in one call.

The request retrieves Germany's total imports and exports to the World
for January, February, and March 2023.

In [5]:
multi_flow_payload = fetch_trade_data(
    type_code="C",
    frequency_code="M",
    classification_code="HS",
    period="202301,202302,202303",
    reporter_code="276",
    partner_code="0",
    commodity_code="TOTAL",
    flow_code="M,X",
    subscription_key=subscription_key,
    max_records=100_000,
)

print("Returned count:", multi_flow_payload["count"])
print("API error:", multi_flow_payload["error"])

for record in multi_flow_payload["data"]:
    print(
        "Period:",
        record["period"],
        "| Flow:",
        record["flowDesc"],
        "| Value:",
        record["primaryValue"],
    )

# To check the unstructured output:
# multi_flow_payload['data']

Returned count: 6
API error: 
Period: 202301 | Flow: Import | Value: 127672961762.962
Period: 202301 | Flow: Export | Value: 139553698569.058
Period: 202302 | Flow: Export | Value: 142244471966.542
Period: 202302 | Flow: Import | Value: 122063479800.551
Period: 202303 | Flow: Export | Value: 163396316089.323
Period: 202303 | Flow: Import | Value: 137664150916.602


## Convert records to a DataFrame

The authenticated API response stores the trade observations inside
the `data` list.

This section converts those records into a pandas DataFrame so that the
returned schema, values, and candidate grain can be inspected.

In [6]:
import pandas as pd

monthly_trade_df = pd.json_normalize(multi_flow_payload['data']) # list of dictionaries -> DataFrame, and flattens nested dictionaries
# otherwise we also could have used pd.DataFrame

print("Shape:", monthly_trade_df.shape)
print(monthly_trade_df.columns.tolist())

monthly_trade_df.head()

Shape: (6, 47)
['typeCode', 'freqCode', 'refPeriodId', 'refYear', 'refMonth', 'period', 'reporterCode', 'reporterISO', 'reporterDesc', 'flowCode', 'flowDesc', 'partnerCode', 'partnerISO', 'partnerDesc', 'partner2Code', 'partner2ISO', 'partner2Desc', 'classificationCode', 'classificationSearchCode', 'isOriginalClassification', 'cmdCode', 'cmdDesc', 'aggrLevel', 'isLeaf', 'customsCode', 'customsDesc', 'mosCode', 'motCode', 'motDesc', 'qtyUnitCode', 'qtyUnitAbbr', 'qty', 'isQtyEstimated', 'altQtyUnitCode', 'altQtyUnitAbbr', 'altQty', 'isAltQtyEstimated', 'netWgt', 'isNetWgtEstimated', 'grossWgt', 'isGrossWgtEstimated', 'cifvalue', 'fobvalue', 'primaryValue', 'legacyEstimationFlag', 'isReported', 'isAggregate']


,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,M,20230101,2023,1,202301,276,DEU,Germany,M,...,None,False,0.0,False,1.276730e+11,NaN,1.276730e+11,0,False,True
1,C,M,20230101,2023,1,202301,276,DEU,Germany,X,...,None,False,0.0,False,NaN,1.395537e+11,1.395537e+11,0,False,True
2,C,M,20230201,2023,2,202302,276,DEU,Germany,X,...,None,False,0.0,False,NaN,1.422445e+11,1.422445e+11,0,False,True
3,C,M,20230201,2023,2,202302,276,DEU,Germany,M,...,None,False,0.0,False,1.220635e+11,NaN,1.220635e+11,0,False,True
4,C,M,20230301,2023,3,202303,276,DEU,Germany,X,...,None,False,0.0,False,NaN,1.633963e+11,1.633963e+11,0,False,True


UN Comtrade recommends:
- exports → FOB-type valuation
- imports → CIF-type valuation
- primaryValue is designed to give the appropriate main analytical trade value depending on what was reported

In [7]:
inspection_columns = [
    "period",
    "reporterDesc",
    "flowCode",
    "flowDesc",
    "partnerDesc",
    "cmdDesc",
    "primaryValue",
    "cifvalue",
    "fobvalue",
]
# Create a view of relevant columns
display(monthly_trade_df[inspection_columns].sort_values(["period", "flowCode"]))
# Check for existence of primary values
print(
    "Missing primary values:",
    monthly_trade_df["primaryValue"]
    .isna()
    .sum()
)
# Check for unique periods, should be 3 based on example
print(
    "Unique periods:",
    sorted(monthly_trade_df["period"].unique())
)
# Check for unique flows, should be 2 based on example
print(
    "Unique flows:",
    sorted(monthly_trade_df["flowCode"].unique())
)

,period,reporterDesc,flowCode,flowDesc,partnerDesc,cmdDesc,primaryValue,cifvalue,fobvalue
0,202301,Germany,M,Import,World,All Commodities,1.276730e+11,1.276730e+11,NaN
1,202301,Germany,X,Export,World,All Commodities,1.395537e+11,NaN,1.395537e+11
3,202302,Germany,M,Import,World,All Commodities,1.220635e+11,1.220635e+11,NaN
2,202302,Germany,X,Export,World,All Commodities,1.422445e+11,NaN,1.422445e+11
5,202303,Germany,M,Import,World,All Commodities,1.376642e+11,1.376642e+11,NaN
4,202303,Germany,X,Export,World,All Commodities,1.633963e+11,NaN,1.633963e+11


Missing primary values: 0
Unique periods: ['202301', '202302', '202303']
Unique flows: ['M', 'X']


## Grain and duplicate validation

For the current monthly-total dataset, one row should represent one
reporter, period, and trade flow combination.

Because partner and commodity are fixed to World and All Commodities,
the candidate grain is:

`reporterCode + period + flowCode`

In [8]:
grain_columns = ["reporterCode", "period", "flowCode"]
# create a boolean series for duplicate checks based on the expected grain. keep=False -> mark every occurence of a duplicate True
duplicate_mask = monthly_trade_df.duplicated(subset=grain_columns, keep=False)
duplicate_rows = monthly_trade_df.loc[
    duplicate_mask, # select the rows where condition is True, meaning duplicates
    grain_columns + [ # results given in the selected columns since df.loc[row_selection, column_selection]
        "reporterDesc",
        "flowDesc",
        "primaryValue"
    ]
]
print("Total rows:", len(monthly_trade_df))
print("Duplicate grain rows:", duplicate_mask.sum()) # total rows where duplicate mask is True
display(duplicate_rows)
# get the number of rows without duplicates, shape[0] ensures we only get the row count
unique_grain_count = monthly_trade_df[grain_columns].drop_duplicates().shape[0]
print("Unique grain combinations:", unique_grain_count)
# Check if the rows are constructed from expected combinations
expected_periods = {
    "202301",
    "202302",
    "202303"
}
expected_flows = {
    "M",
    "X"
}
# Create a set of expected combinations
expected_combinations = {(period, flow) for period in expected_periods for flow in expected_flows}
# expected_combinations = set() # longer version
# for period in expected_periods:
#     for flow in expected_flows:
#         expected_combinations.add(
#             (period, flow)
#         )
# Get the available combinations in the monthly df. .itertuples() is used to convert rows into tuples, index=False: don't include index in tuple,
returned_combinations = set(monthly_trade_df[["period", "flowCode"]].itertuples(index=False, name=None)) # name=none for non named tuples
missing_combinations = expected_combinations - returned_combinations
unexpected_combinations = returned_combinations - expected_combinations
print("Missing combinations:", missing_combinations)
print("Unexpected combinations:", unexpected_combinations)

Total rows: 6
Duplicate grain rows: 0


,reporterCode,period,flowCode,reporterDesc,flowDesc,primaryValue


Unique grain combinations: 6
Missing combinations: set()
Unexpected combinations: set()


## Monthly Trade Summary

The API returns imports and exports as separate rows.

This section reshapes the validated monthly trade data so that import
and export values appear as separate columns for each reporter and period.

In [9]:
monthly_summary_df = monthly_trade_df.pivot_table(
        index=["reporterCode", "reporterDesc", "period"], # defines what stays as the row identity, reporter x period x flow -> reporter x period
        columns="flowDesc", # Import, Export become column names
        values="primaryValue", # Numbers to be placed under new columns
        aggfunc="sum", # decides how to handle duplicate rows, e.g. Germany x 202301 x Export has more than 1 value in table
    ).reset_index() # without resetting index, index columns become actual df index levels, with reset index they reamin ordinary columns
# After pivoting, pandas may name the column axis with 'columns' value, in this case 'flowDesc'
monthly_summary_df.columns.name = None
monthly_summary_df["tradeBalance"] = monthly_summary_df["Export"] - monthly_summary_df["Import"]
display(monthly_summary_df.sort_values("period"))

,reporterCode,reporterDesc,period,Export,Import,tradeBalance
0,276,Germany,202301,1.395537e+11,1.276730e+11,1.188074e+10
1,276,Germany,202302,1.422445e+11,1.220635e+11,2.018099e+10
2,276,Germany,202303,1.633963e+11,1.376642e+11,2.573217e+10


In [10]:
# Validation Checks
print("Summary rows:", len(monthly_summary_df))
print("Unique periods:", monthly_summary_df["period"].nunique())
print("Missing imports:", monthly_summary_df["Import"].isna().sum())
print("Missing exports:", monthly_summary_df["Export"].isna().sum())
print("Missing trade balances:", monthly_summary_df["tradeBalance"].isna().sum())

Summary rows: 3
Unique periods: 3
Missing imports: 0
Missing exports: 0
Missing trade balances: 0


In [11]:
# Readability Improvements
monthly_summary_df["Import_Bn"] = monthly_summary_df["Import"] / 1_000_000_000
monthly_summary_df["Export_Bn"] = monthly_summary_df["Export"] / 1_000_000_000
monthly_summary_df["tradeBalance_Bn"] = monthly_summary_df["tradeBalance"] / 1_000_000_000
display(monthly_summary_df[["reporterDesc", "period", "Import_Bn", "Export_Bn", "tradeBalance_Bn"]].round(2))

,reporterDesc,period,Import_Bn,Export_Bn,tradeBalance_Bn
0,Germany,202301,127.67,139.55,11.88
1,Germany,202302,122.06,142.24,20.18
2,Germany,202303,137.66,163.40,25.73


## Metadata Validation

UN Comtrade trade values are reported in current US dollars.

This section queries the authenticated metadata endpoint for Germany's
monthly HS data in January 2023 to inspect reporter-specific metadata,
publication notes, and other contextual information.

In [14]:
from src.extract import fetch_trade_metadata

metadata_payload = fetch_trade_metadata(
    type_code="C",
    frequency_code="M",
    classification_code="HS",
    period="202301",
    reporter_code="276",
    subscription_key=subscription_key,
)

print(type(metadata_payload))
print(metadata_payload.keys())

metadata_payload

<class 'dict'>
dict_keys(['elapsedTime', 'count', 'data', 'error'])


{'elapsedTime': '0.04 secs',
 'count': 1,
 'data': [{'reporterCode': 276,
   'period': 202301,
   'typeCode': 'C',
   'freqCode': 'M',
   'datasetCode': 30276202301202100,
   'notes': [{'datasetCode': 30276202301202100,
     'typeCode': 'C',
     'freqCode': 'M',
     'period': 202301,
     'reporterCode': 276,
     'reporterDescription': 'Germany',
     'currency': 'EUR',
     'importConvFactor': '1.0769',
     'exportConvFactor': '1.0769',
     'tradeSystem': 'Special',
     'classificationCode': 'H6',
     'importValuation': 'CIF',
     'exportValuation': 'FOB',
     'importPartnerCountry': 'Origin',
     'exportPartnerCountry': 'Last Known Destination',
     'importPartner2Country': 'Consignment',
     'exportPartner2Country': 'N/A',
     'publicationNote': 'New Data: Data for this period is published for the first time; the new data was collected from Federal Statistical Office (Destatis) on 13 Apr 23.',
     'publicationDate': '2023-04-29T02:05:17.163',
     'publicationDateShort

In [15]:
# metadata currency = EUR, trade values in API records = converted USD values
# currency = EUR, importConvFactor = 1.0769, exportConvFactor = 1.0769
# importValuation = CIF means imports are valued on a CIF basis: cost + insurance + freight.
# exportValuation = FOB means exports are valued on an FOB basis: value at the exporting border, excluding later international transport costs.

metadata_record = metadata_payload["data"][0]
latest_note = metadata_record["notes"][-1]

print("Reported currency:", latest_note["currency"])
print("Import conversion factor:", latest_note["importConvFactor"])
print("Export conversion factor:", latest_note["exportConvFactor"])
print("Import valuation:", latest_note["importValuation"])
print("Export valuation:", latest_note["exportValuation"])
print("Latest publication date:", latest_note["publicationDateShort"])
print("Latest publication note:", latest_note["publicationNote"])

Reported currency: EUR
Import conversion factor: 1.0769
Export conversion factor: 1.0769
Import valuation: CIF
Export valuation: FOB
Latest publication date: 2025-07-26
Latest publication note: Revised Data: Data for this period was published before; the revised data was collected from Federal Statistical Office (Destatis) on 27 Feb 25.
